# Data Reliability - 数据质量

> **适用场景**: 数据平台可靠性保障、数据治理
> **面试频率**: ⭐⭐⭐⭐⭐ 极高频（Senior DE 必考）

## 目录
1. Data Contract 设计
2. Freshness / Completeness 指标
3. Schema Breaking Change 处理
4. Upstream Failure 防护
5. 练习题

---
## 1. Data Contract 设计

### 什么是 Data Contract？
Data Contract 是数据生产者和消费者之间的**正式协议**，定义了数据的：
- **Schema**：字段名、类型、是否可空
- **语义**：字段含义、计算逻辑、业务规则
- **质量承诺**：SLA（延迟、完整性、准确性）
- **变更流程**：Breaking change 通知机制

### Data Contract 示例（YAML 格式）
```yaml
# contracts/orders.yaml
version: 1.0.0
name: orders
owner: data-platform-team
consumers:
  - analytics-team
  - ml-team

schema:
  - name: order_id
    type: STRING
    nullable: false
    description: "全局唯一订单号，格式 ORD-{timestamp}-{uuid}"
  - name: customer_id
    type: STRING
    nullable: false
  - name: total_amount_usd
    type: DECIMAL(10,2)
    nullable: false
    description: "含税总价，单位美元"
  - name: status
    type: STRING
    nullable: false
    allowed_values: ["pending", "paid", "shipped", "delivered", "cancelled"]
  - name: created_at
    type: TIMESTAMP
    nullable: false

sla:
  freshness: 30 minutes      # 数据延迟上限
  completeness: 99.9%         # 数据完整性
  availability: 99.5%         # 表可查询可用率

quality_rules:
  - rule: order_id IS UNIQUE
  - rule: total_amount_usd > 0
  - rule: created_at >= '2020-01-01'

breaking_change_policy:
  notice_period: 30 days
  notification_channel: slack:#data-contracts
```

### Data Contract 实施工具
- **soda-core**：数据质量检查框架，支持 Contract 定义
- **dbt tests + schema.yml**：轻量级 contract 实施
- **Atlan / DataHub**：数据目录 + Contract 管理
- **OpenDataContract（开源标准）**：YAML-based contract 规范

---
## 2. Freshness / Completeness 指标

### Freshness（新鲜度）
衡量数据的**及时性**：从数据产生到可查询的延迟。

```sql
-- 检查数据新鲜度：最新数据距现在多久
SELECT
    CURRENT_TIMESTAMP - MAX(created_at) AS data_lag,
    MAX(created_at) AS latest_record
FROM orders;

-- 告警：若超过 30 分钟无新数据
-- dbt source freshness 配置
-- freshness:
--   warn_after: {count: 30, period: minute}
--   error_after: {count: 60, period: minute}
```

```python
# 用 Great Expectations 检查 freshness
import great_expectations as ge

context = ge.get_context()
# 检查最新行的时间戳不超过1小时前
result = context.run_checkpoint(
    checkpoint_name='freshness_check',
    # 内部验证：MAX(event_time) >= NOW() - INTERVAL '1 hour'
)
```

### Completeness（完整性）
衡量数据的**完整程度**：预期行数 vs 实际行数，非空率等。

```sql
-- 1. 行数完整性：今天的订单数是否在合理范围内
WITH today AS (
    SELECT COUNT(*) AS cnt FROM orders
    WHERE DATE(created_at) = CURRENT_DATE
),
baseline AS (
    -- 过去4周同一天的平均行数
    SELECT AVG(daily_cnt) AS avg_cnt, STDDEV(daily_cnt) AS std_cnt
    FROM (
        SELECT DATE(created_at) AS d, COUNT(*) AS daily_cnt
        FROM orders
        WHERE DATE(created_at) BETWEEN CURRENT_DATE - 28 AND CURRENT_DATE - 1
          AND EXTRACT(DOW FROM created_at) = EXTRACT(DOW FROM CURRENT_DATE)
        GROUP BY 1
    )
)
SELECT
    today.cnt,
    baseline.avg_cnt,
    ABS(today.cnt - baseline.avg_cnt) / NULLIF(baseline.std_cnt, 0) AS z_score
    -- z_score > 3 → 异常告警
FROM today, baseline;

-- 2. 字段非空率
SELECT
    COUNT(*) AS total,
    COUNT(email) AS email_not_null,
    COUNT(email)::FLOAT / COUNT(*) AS email_completeness
FROM customers;
```

### 其他数据质量维度
| 维度 | 含义 | 检查方法 |
|------|------|----------|
| **Accuracy** | 数据是否正确反映真实情况 | 与源系统对账 |
| **Uniqueness** | 主键/唯一键是否重复 | `COUNT(*) vs COUNT(DISTINCT key)` |
| **Validity** | 值是否在允许范围内 | 正则、枚举值检查 |
| **Consistency** | 跨表/跨系统数据是否一致 | 外键完整性、跨源对账 |

---
## 3. Schema Breaking Change 处理

### 什么是 Breaking Change？
- 删除列
- 重命名列
- 改变列类型（尤其是不兼容转换）
- 改变主键
- NOT NULL 约束添加到已有可空列

### 处理策略

**策略 1：版本化（推荐）**
```sql
-- 保留旧表，创建新版本
CREATE VIEW orders_v2 AS
SELECT
    order_id,
    client_id AS customer_id,  -- 重命名：保持旧名兼容，同时暴露新名
    total_amount_cents / 100.0 AS total_amount_usd  -- 类型/单位变更
FROM orders_raw;

-- 迁移计划
-- Phase 1: 提前30天通知消费者
-- Phase 2: 新版本上线，两个版本并存
-- Phase 3: 消费者迁移完成后，废弃旧版本
```

**策略 2：向后兼容演进（Parquet/Avro Schema Evolution）**
```python
# Avro schema 演进规则
# ✅ 安全：新增有默认值的字段
# ✅ 安全：删除有默认值的字段（读旧数据时使用默认值）
# ❌ 危险：修改已有字段类型
# ❌ 危险：删除没有默认值的字段

# Delta Lake schema evolution
df.write \
    .option('mergeSchema', 'true') \  # 允许新增列
    .format('delta') \
    .mode('append') \
    .save('/path/to/table')
```

**策略 3：Contract First**
- 所有 schema 变更先提 PR 修改 Data Contract
- CI 检查变更是否 breaking（使用 `schemadiff` 等工具）
- 需要审批和消费者确认才能合并

---
## 4. Upstream Failure 防护

### 常见上游故障模式
1. **数据延迟**：上游 ETL 比预期晚到
2. **数据缺失**：上游表某些分区为空
3. **Schema 变更**：上游字段被删除/重命名
4. **数据质量下降**：上游数据大量 NULL 或异常值
5. **上游服务不可用**：API/数据库连接失败

### 防护模式

**模式 1：传感器检查（Sensor）**
```python
# Airflow ExternalTaskSensor
wait_for_upstream = ExternalTaskSensor(
    task_id='wait_for_orders',
    external_dag_id='orders_pipeline',
    external_task_id='load_to_warehouse',
    timeout=3600,           # 最多等1小时
    poke_interval=60,       # 每分钟检查一次
    mode='reschedule',      # 释放 worker，不占资源
)

# 自定义 Sensor：检查分区存在
check_partition = BigQueryTablePartitionExistenceSensor(
    task_id='check_orders_partition',
    project_id='my-project',
    dataset_id='raw',
    table_id='orders',
    partition_id='{{ ds_nodash }}',  # 今天的分区
)
```

**模式 2：数据质量门控**
```python
# 在 DAG 中加质量检查，不通过则停止下游
from great_expectations_provider.operators.great_expectations import GreatExpectationsOperator

validate_orders = GreatExpectationsOperator(
    task_id='validate_orders',
    checkpoint_name='orders_checkpoint',
    fail_task_on_validation_failure=True,  # 质量不达标则 DAG 失败
)
```

**模式 3：隔离影响（Quarantine Pattern）**
```sql
-- 将质量不合格的数据写入隔离区，而非直接失败
-- 好数据 → 正式表
INSERT INTO orders_clean
SELECT * FROM orders_staging
WHERE order_id IS NOT NULL AND total_amount > 0;

-- 坏数据 → 隔离表（保留供排查）
INSERT INTO orders_quarantine
SELECT *, CURRENT_TIMESTAMP AS quarantine_at, 'invalid_amount' AS reason
FROM orders_staging
WHERE total_amount <= 0;
```

**模式 4：幂等设计**
- 上游重跑不会产生重复数据
- 使用 `MERGE/UPSERT` 而非直接 `INSERT`
- 按日期分区覆盖（partition overwrite）

---
## 5. 练习题

### Q1 [高频] 什么是 Data Contract？为什么越来越重要？

<details><summary>参考答案</summary>

Data Contract 是数据生产者与消费者之间的正式协议，明确定义数据的 schema、语义、质量 SLA 和变更流程。

重要性：
1. **组织规模增大**：多团队共享数据时，无 contract 导致下游频繁被上游变更破坏
2. **数据网格（Data Mesh）架构**：每个 domain 作为数据产品需要对消费者承诺
3. **可观测性**：contract 是监控的基准（偏离 contract = 告警）
4. **责任清晰**：明确谁负责保证数据质量
</details>

---

### Q2 [高频] 如何检测数据量异常（突然减少 50%）？

<details><summary>参考答案</summary>

**统计方法**：计算 z-score（今日行数与历史同期的标准差倍数）
- |z-score| > 2 → 警告
- |z-score| > 3 → 错误告警

```sql
-- 今日行数 vs 过去4周同星期几均值
WITH history AS (
    SELECT AVG(cnt) AS avg_cnt, STDDEV(cnt) AS std_cnt
    FROM daily_row_counts
    WHERE date BETWEEN CURRENT_DATE-28 AND CURRENT_DATE-1
      AND day_of_week = EXTRACT(DOW FROM CURRENT_DATE)
),
today AS (SELECT COUNT(*) AS cnt FROM orders WHERE DATE(created_at) = CURRENT_DATE)
SELECT
    today.cnt,
    (today.cnt - history.avg_cnt) / NULLIF(history.std_cnt, 0) AS z_score
FROM today, history;
```

**工具**：Monte Carlo, Bigeye, dbt-expectations 的 `expect_table_row_count_to_be_between`。
</details>

---

### Q3 上游 schema 发生 Breaking Change 而你没有提前收到通知，你如何处理？

<details><summary>参考答案</summary>

**立即处置**：
1. 暂停下游 pipeline，避免传播损坏数据
2. 联系上游团队确认变更内容和时间
3. 评估影响范围：哪些表/报表受影响？

**修复**：
- 如果是字段重命名：在 staging 层做映射兼容
- 如果是类型变更：转换后再处理
- 历史数据是否需要 backfill？

**事后改进**：
1. 建立 Schema Registry（如 Confluent Schema Registry 或 dbt source 的 schema test）
2. CI 中加 schema diff 检查
3. 推动上游团队签署 Data Contract，明确变更通知义务
4. 在 pipeline 入口加 schema 验证（发现变更立刻告警而非静默失败）
</details>

---

### Q4 Freshness 和 Completeness 的区别？如何分别监控？

<details><summary>参考答案</summary>

- **Freshness（新鲜度）**：时间维度，数据多旧？`NOW() - MAX(event_time)` 来衡量延迟
- **Completeness（完整性）**：数量维度，数据多全？实际行数/非空率是否符合预期

**监控方式**：
- Freshness：定时检查 `MAX(load_time)` 或 `MAX(event_time)`，超过阈值告警
- Completeness：
  - 列级：`COUNT(col)/COUNT(*) >= 0.99`
  - 行级：今日行数 / 昨日行数的比率在合理范围（0.8~1.2）
  - 端到端：与源系统对账（source record count vs warehouse count）
</details>

---

### Q5 [实战] 设计一个数据质量监控框架，要包含哪些要素？

<details><summary>参考答案</summary>

**核心要素**：
1. **规则引擎**：定义和存储质量规则（freshness、completeness、uniqueness、validity）
2. **执行层**：定期运行规则，与 pipeline 集成（dbt test、Great Expectations、Soda）
3. **存储**：历史质量指标存储（用于趋势分析和阈值学习）
4. **告警**：多级告警（warn/error），对接 PagerDuty/Slack
5. **Dashboard**：可视化质量趋势，per-table 的质量评分
6. **血缘集成**：知道质量问题影响哪些下游
7. **Incident 追踪**：与 Jira/Linear 集成，质量事件自动创建 ticket

**工具选型**：
- 开源：dbt tests + Elementary（dbt 质量可视化）
- 商业：Monte Carlo、Bigeye、Acceldata
</details>